## Make dataset out of Hugginggface dataset

In [1]:
import os
from tqdm import tqdm
from datasets import load_dataset
import json
import google.generativeai as genai
from PIL import Image
import time

/home/ray/dla/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ray/dla/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/home/ray/dla/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Pytho

In [8]:
dataset = load_dataset("dpdl-benchmark/oxford_flowers102")

train_dataset = dataset["train"]
fid_train_dataset = dataset["test"]
fid_test_dataset = dataset["validation"]

In [ ]:
shuffled_test = fid_test_dataset.shuffle(seed=42)

blip_test_dataset = shuffled_test.select(range(len(shuffled_test)//3))
llava_test_dataset = shuffled_test.select(range(len(shuffled_test)//3, 2*len(shuffled_test)//3))
gemini_test_dataset = shuffled_test.select(range(2*len(shuffled_test)//3, len(shuffled_test)))

In [3]:
from dotenv import load_dotenv

load_dotenv()

# Configuration de l'API Gemini
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
genai.configure(api_key=GOOGLE_API_KEY)

# Initialisation du modèle Gemini 2.5 Pro
model = genai.GenerativeModel('gemini-2.5-pro')

def generate_caption_with_gemini(image, max_retries=3, delay=1):
    """Génère une légende pour une image en utilisant Gemini 3.0 Pro avec retry"""
    prompt = """
Describe this image of a flower and its style in a very detailed manner. Describe the color, shape, and texture of the petals, the center of the flower, the leaves, and the background. Mention the lighting and the photography style (e.g. close-up, bokeh). Context: This description will be used as a ground-truth caption to train a Text-to-Image diffusion model.
Constraint: Focus ONLY on visual, physical details. Do not use poetic language.
Output ONLY the description text, nothing else.
"""
    
    for attempt in range(max_retries):
        try:
            response = model.generate_content([prompt, image])            
            return response.text.strip()
        except Exception as e:
            if attempt < max_retries - 1:
                print(
                    f"Error generating caption (attempt {attempt + 1}/{max_retries}): "
                    f"{e}. Retrying in {delay}s..."
                )
                time.sleep(delay)
            else:
                raise e

# Train dataset

In [ ]:
root_dir = "./datasets/oxford_flowers102-gemini"
images_dir = "images"
captions_dir = "captions"

images_dir_absolute = os.path.join(root_dir, images_dir)
captions_dir_absolute = os.path.join(root_dir, captions_dir)

if not os.path.exists(root_dir):
    os.makedirs(os.path.join(root_dir, images_dir))

if not os.path.exists(os.path.join(root_dir, images_dir)):
    os.makedirs(os.path.join(root_dir, images_dir))
if not os.path.exists(os.path.join(root_dir, captions_dir)):
    os.makedirs(os.path.join(root_dir, captions_dir))

image_format = "png"
json_name = "partition/data_info.json"
if not os.path.exists(os.path.join(root_dir, "partition")):
    os.makedirs(os.path.join(root_dir, "partition"))

absolute_json_name = os.path.join(root_dir, json_name)

# Reconstruire data_info à partir des fichiers existants
data_info = []
existing_captions = sorted([int(f.replace('.txt', '')) for f in os.listdir(captions_dir_absolute) if f.endswith('.txt')])

if existing_captions:
    print(f"Reconstruction de data_info à partir de {len(existing_captions)} captions existantes...")
    for idx in existing_captions:
        with open(f"{captions_dir_absolute}/{idx}.txt", "r") as text_file:
            prompt = text_file.read()
        
        # Charger l'image pour obtenir ses vraies dimensions
        img = Image.open(f"{images_dir_absolute}/{idx}.{image_format}")
        width, height = img.size
        ratio = width / height
        
        data_info.append({
            "height": height,
            "width": width,
            "ratio": ratio,
            "path": f"images/{idx}.{image_format}",
            "prompt": prompt,
        })
    
    start_order = existing_captions[-1] + 1
    print(f"Dernière caption trouvée: {existing_captions[-1]}, reprise à l'image {start_order}")
else:
    start_order = 0
    print("Démarrage depuis le début")

order = start_order
for item in tqdm(
    train_dataset.select(range(start_order, len(train_dataset))),
    initial=start_order,
    total=len(train_dataset)
): 
    image = item["image"]
    image.save(f"{images_dir_absolute}/{order}.{image_format}")
    
    # Générer le label avec Gemini au lieu d'utiliser le label du dataset
    label_name = generate_caption_with_gemini(image)
    
    with open(f"{captions_dir_absolute}/{order}.txt", "w") as text_file:
        text_file.write(label_name)
    
    # Utiliser les vraies dimensions de l'image
    width, height = image.size
    ratio = width / height
    
    data_info.append({
        "height": height,
        "width": width,
        "ratio": ratio,
        "path": f"images/{order}.{image_format}",
        "prompt": label_name,
    })
    
    # Sauvegarder le JSON après chaque image (checkpoint)
    with open(absolute_json_name, "w") as json_file:
        json.dump(data_info, json_file)
        
    order += 1

print(f"\nTerminé ! {order} images traitées au total")

# Test subset dataset

In [ ]:
root_dir = "./datasets/oxford_flowers102-gemini-test"
images_dir = "images"
captions_dir = "captions"

images_dir_absolute = os.path.join(root_dir, images_dir)
captions_dir_absolute = os.path.join(root_dir, captions_dir)

if not os.path.exists(root_dir):
    os.makedirs(os.path.join(root_dir, images_dir))

if not os.path.exists(os.path.join(root_dir, images_dir)):
    os.makedirs(os.path.join(root_dir, images_dir))
if not os.path.exists(os.path.join(root_dir, captions_dir)):
    os.makedirs(os.path.join(root_dir, captions_dir))

image_format = "png"
json_name = "partition/data_info.json"
if not os.path.exists(os.path.join(root_dir, "partition")):
    os.makedirs(os.path.join(root_dir, "partition"))

absolute_json_name = os.path.join(root_dir, json_name)

# Reconstruire data_info à partir des fichiers existants
data_info = []
existing_captions = sorted([int(f.replace('.txt', '')) for f in os.listdir(captions_dir_absolute) if f.endswith('.txt')])

if existing_captions:
    print(f"Reconstruction de data_info à partir de {len(existing_captions)} captions existantes...")
    for idx in existing_captions:
        with open(f"{captions_dir_absolute}/{idx}.txt", "r") as text_file:
            prompt = text_file.read()
        
        # Charger l'image pour obtenir ses vraies dimensions
        img = Image.open(f"{images_dir_absolute}/{idx}.{image_format}")
        width, height = img.size
        ratio = width / height
        
        data_info.append({
            "height": height,
            "width": width,
            "ratio": ratio,
            "path": f"images/{idx}.{image_format}",
            "prompt": prompt,
        })
    
    start_order = existing_captions[-1] + 1
    print(f"Dernière caption trouvée: {existing_captions[-1]}, reprise à l'image {start_order}")
else:
    start_order = 0
    print("Démarrage depuis le début")

order = start_order
for item in tqdm(
    gemini_test_dataset.select(range(start_order, len(gemini_test_dataset))), 
    initial=start_order, 
    total=len(gemini_test_dataset)
): 
    image = item["image"]
    image.save(f"{images_dir_absolute}/{order}.{image_format}")
    
    # Générer le label avec Gemini au lieu d'utiliser le label du dataset
    label_name = generate_caption_with_gemini(image)
    
    with open(f"{captions_dir_absolute}/{order}.txt", "w") as text_file:
        text_file.write(label_name)
    
    # Utiliser les vraies dimensions de l'image
    width, height = image.size
    ratio = width / height
    
    data_info.append({
        "height": height,
        "width": width,
        "ratio": ratio,
        "path": f"images/{order}.{image_format}",
        "prompt": label_name,
    })
    
    # Sauvegarder le JSON après chaque image (checkpoint)
    with open(absolute_json_name, "w") as json_file:
        json.dump(data_info, json_file)
        
    order += 1

print(f"\nTerminé ! {order} images traitées au total")